# Notebook for Library merging

In [1]:
# Imports
from itertools import product
from sklearn.model_selection import ParameterGrid
from Models.config_parser import load_config, load_dataset_config
from Models.dataset_and_models import Dataset, Model

In [2]:
# Load configurations
CONFIG = "../config.yaml"

model_config   = load_config(CONFIG)        # {model_key: {param: [values]}}
dataset_config = load_dataset_config(CONFIG) # {dataset_key: {n_features: [...], n_samples: [...]}}

In [3]:
# All model × hyperparameter combinations
model_runs = [
    (model_key, params)
    for model_key, param_grid in model_config.items()
    for params in ParameterGrid(param_grid)
]

# All dataset × (n_features, n_samples) combinations
dataset_runs = [
    (dataset_key, params)
    for dataset_key, param_grid in dataset_config.items()
    for params in ParameterGrid(param_grid)
]


In [4]:
# Full cross-product: one entry per benchmark run
benchmark_runs = list(product(model_runs, dataset_runs))
print(f"{len(model_runs)} model configs × {len(dataset_runs)} dataset configs = {len(benchmark_runs)} total runs")

# Load a single dataset variant
dataset_key, dataset_params = dataset_runs[0]
ds = Dataset[dataset_key.upper()].load_dataset(**dataset_params)
X, y = ds["X"], ds["y"]

58 model configs × 324 dataset configs = 18792 total runs


### Iterate trough every combination and train the model for explanation

In [ ]:
from tqdm import tqdm
resulting_models = []
for dataset_key, dataset_params in tqdm(dataset_runs):
    dataset_enum = Dataset[dataset_key.upper()]
    ds = dataset_enum.load_dataset(**dataset_params)
    X, y = ds["X"], ds["y"]
    for model_key, model_params in tqdm(model_runs, leave=False):
        trainer = Model[model_key.upper()].get_model_with_params(dataset_enum, model_params)
        trainer.fit(X, y, task=ds["task"])
        resulting_models.append(trainer.get_model())

  0%|          | 0/324 [00:05<?, ?it/s]


KeyboardInterrupt: 